# Apriori Weather Patterns (Legacy)

**Source script:** `apriori_weather_patterns.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import logging
import re
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError(
        "matplotlib is required. Install with: pip install matplotlib"
    ) from exc

try:
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError as exc:
    raise ImportError(
        "mlxtend is required. Install with: pip install mlxtend"
    ) from exc


SEED = 42

ENRICHED_PATH = "outputs/imputed_dataset_enriched.csv"
OUTPUT_DIR = Path("outputs/apriori")

DEFAULT_MIN_SUPPORT = 0.10
SENSITIVITY_SUPPORTS = [0.08, 0.10, 0.12]
MIN_CONFIDENCE = 0.60
MIN_LIFT = 1.20

PRUNE_CONF_DIFF = 0.02
PRUNE_LIFT_DIFF = 0.05


def setup_logging() -> logging.Logger:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )
    return logging.getLogger("apriori_weather_patterns")


def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def sanitize_token(value: object) -> str:
    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "unknown"


def itemset_to_str(items: Iterable[str]) -> str:
    vals = sorted([str(x) for x in items])
    return "{" + ", ".join(vals) + "}"


def rule_key(antecedents: Iterable[str], consequents: Iterable[str]) -> str:
    return f"{itemset_to_str(antecedents)} -> {itemset_to_str(consequents)}"


def to_markdown_table(df: pd.DataFrame, columns: Sequence[str]) -> str:
    if df.empty:
        return "_No rows available._"
    headers = list(columns)
    rows = [headers, ["---"] * len(headers)]
    for _, row in df[headers].iterrows():
        rows.append([str(row[c]) for c in headers])
    return "\n".join("| " + " | ".join(r) + " |" for r in rows)


def resolve_output_path(output_dir: Path, filename: str) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    base = output_dir / filename
    if not base.exists():
        return base
    stem = base.stem
    suffix = base.suffix
    i = 2
    while True:
        candidate = output_dir / f"{stem}_v{i}{suffix}"
        if not candidate.exists():
            return candidate
        i += 1


def infer_season(month: int) -> str:
    if month in (12, 1, 2):
        return "winter"
    if month in (3, 4, 5):
        return "spring"
    if month in (6, 7, 8):
        return "summer"
    return "autumn"


def load_data(enriched_path: str = ENRICHED_PATH) -> pd.DataFrame:
    path = Path(enriched_path)
    if not path.exists():
        alt = Path("outputs/imputed_dataset_enriched.xlsx")
        if alt.exists():
            path = alt
        else:
            raise FileNotFoundError(
                f"Enriched dataset not found. Checked: {path} and {alt}"
            )

    ext = path.suffix.lower()
    if ext == ".csv":
        df = pd.read_csv(path)
    elif ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        raise ValueError(
            f"Unsupported input format: {path.suffix}. Use .csv or .xlsx/.xls"
        )
    return df


def detect_date_column(columns: Iterable[str]) -> Optional[str]:
    candidates = [
        "presentation_date",
        "date of presentation",
        "admission_date",
        "date",
        "date_of_presentation",
    ]
    col_list = list(columns)
    norm_map = {normalize_name(c): c for c in col_list}
    for c in candidates:
        if c in col_list:
            return c
        c_norm = normalize_name(c)
        if c_norm in norm_map:
            return norm_map[c_norm]
    for c in col_list:
        if "date" in str(c).lower():
            return c
    return None


def detect_outcome_column(df: pd.DataFrame) -> str:
    candidates = [
        "diagnosis_group",
        "group",
        "(suspected) diagnosis",
        "diagnosis",
        "outcome_group",
        "outcome",
        "clinical course",
    ]
    cols = list(df.columns)
    norm_map = {normalize_name(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        c_norm = normalize_name(c)
        if c_norm in norm_map:
            return norm_map[c_norm]

    fallback = []
    for c in cols:
        c_norm = normalize_name(c)
        if any(tok in c_norm for tok in ["outcome", "diagnosis", "group"]):
            if pd.api.types.is_object_dtype(df[c]) or str(df[c].dtype) == "category":
                fallback.append(c)
    if fallback:
        return fallback[0]
    raise ValueError(
        "Could not detect outcome group column. "
        "Add one of: diagnosis_group, group, diagnosis, outcome_group."
    )


def detect_weather_numeric_columns(df: pd.DataFrame, exclude_cols: Set[str]) -> List[str]:
    weather_tokens = [
        "temperature",
        "wind",
        "precip",
        "rain",
        "snow",
        "radiation",
        "uv",
        "daylight",
        "sunshine",
        "pressure",
        "et0",
        "evapotranspiration",
    ]
    cols = []
    for col in df.columns:
        if col in exclude_cols:
            continue
        name = str(col).lower()
        if "__missing" in name:
            continue
        if "moon" in name:
            continue
        if not any(tok in name for tok in weather_tokens):
            continue
        series_num = pd.to_numeric(df[col], errors="coerce")
        if series_num.notna().sum() >= 10:
            cols.append(col)
    return cols


def discretize_quantiles(
    series: pd.Series,
    q: int,
    labels_3: Optional[List[str]] = None,
) -> pd.Series:
    series_num = pd.to_numeric(series, errors="coerce")
    valid = series_num.dropna()
    out = pd.Series(index=series.index, dtype="object")
    if valid.nunique() < 2:
        return out
    try:
        binned = pd.qcut(series_num, q=q, duplicates="drop")
    except ValueError:
        return out

    cats = list(binned.cat.categories)
    if not cats:
        return out

    if q == 3:
        if len(cats) == 1:
            labels = ["mid"]
        elif len(cats) == 2:
            labels = ["low", "high"]
        else:
            labels = labels_3 if labels_3 else ["low", "mid", "high"]
            labels = labels[: len(cats)]
    elif q == 4:
        labels = [f"q{i+1}" for i in range(len(cats))]
    else:
        labels = [f"q{i+1}" for i in range(len(cats))]

    mapping = {cat: labels[i] for i, cat in enumerate(cats)}
    out = binned.map(mapping).astype("object")
    return out


def add_boolean_items_from_category(
    transactions: pd.DataFrame,
    category_series: pd.Series,
    prefix: str,
    allowed_values: Optional[List[str]] = None,
) -> List[str]:
    created_items: List[str] = []
    s = category_series.astype("object")
    values = allowed_values if allowed_values else sorted(v for v in s.dropna().unique())
    used_names: Set[str] = set(transactions.columns)

    for raw_val in values:
        val_name = sanitize_token(raw_val)
        item = f"{prefix}_{val_name}"
        suffix = 2
        while item in used_names:
            item = f"{prefix}_{val_name}_{suffix}"
            suffix += 1
        transactions[item] = (s == raw_val).fillna(False)
        used_names.add(item)
        created_items.append(item)
    return created_items


def build_transactions(df: pd.DataFrame) -> Dict[str, object]:
    outcome_col = detect_outcome_column(df)
    date_col = detect_date_column(df.columns)
    exclude_cols = {outcome_col}
    if date_col:
        exclude_cols.add(date_col)

    transactions = pd.DataFrame(index=df.index)
    weather_items: Set[str] = set()
    temporal_items: Set[str] = set()
    moon_items: Set[str] = set()
    outcome_items: Set[str] = set()

    weather_numeric_cols = detect_weather_numeric_columns(df, exclude_cols)
    weather_discretized = 0

    for col in weather_numeric_cols:
        labels = discretize_quantiles(df[col], q=3, labels_3=["low", "mid", "high"])
        if labels.notna().sum() == 0:
            continue
        prefix = sanitize_token(col)
        created = add_boolean_items_from_category(
            transactions=transactions,
            category_series=labels,
            prefix=prefix,
            allowed_values=[v for v in ["low", "mid", "high"] if (labels == v).any()],
        )
        weather_items.update(created)
        weather_discretized += 1

    moon_cols = [c for c in df.columns if "moon" in str(c).lower()]
    moon_numeric_count = 0
    moon_categorical_count = 0

    for col in moon_cols:
        prefix = sanitize_token(col)
        if pd.api.types.is_numeric_dtype(df[col]) or pd.to_numeric(df[col], errors="coerce").notna().sum() > 0:
            labels = discretize_quantiles(df[col], q=4)
            if labels.notna().sum() > 0:
                created = add_boolean_items_from_category(transactions, labels, prefix)
                moon_items.update(created)
                moon_numeric_count += 1
            continue

        cat = df[col].astype("object").fillna("missing").map(lambda x: sanitize_token(x))
        created = add_boolean_items_from_category(transactions, cat, prefix)
        moon_items.update(created)
        moon_categorical_count += 1

    dt_series = None
    if date_col:
        dt_series = pd.to_datetime(df[date_col], errors="coerce")

    if dt_series is not None and dt_series.notna().any():
        month_series = dt_series.dt.month.map(
            {
                1: "jan",
                2: "feb",
                3: "mar",
                4: "apr",
                5: "may",
                6: "jun",
                7: "jul",
                8: "aug",
                9: "sep",
                10: "oct",
                11: "nov",
                12: "dec",
            }
        )
        weekday_series = dt_series.dt.day_name().str[:3].str.lower()
        season_series = dt_series.dt.month.map(lambda m: infer_season(int(m)) if pd.notna(m) else np.nan)

        temporal_items.update(add_boolean_items_from_category(transactions, month_series, "month"))
        temporal_items.update(add_boolean_items_from_category(transactions, weekday_series, "weekday"))
        temporal_items.update(add_boolean_items_from_category(transactions, season_series, "season"))
    else:
        for col in df.columns:
            n = normalize_name(col)
            if n in {"month", "weekday", "dayofweek", "season"}:
                cat = df[col].astype("object").fillna("missing").map(lambda x: sanitize_token(x))
                temporal_items.update(add_boolean_items_from_category(transactions, cat, sanitize_token(col)))

    outcome_series_raw = df[outcome_col].astype("object").fillna("missing")
    outcome_series = outcome_series_raw.map(lambda x: sanitize_token(x))
    outcome_items.update(add_boolean_items_from_category(transactions, outcome_series, "outcome"))

    outcome_prev = (
        outcome_series_raw.fillna("missing")
        .value_counts(dropna=False)
        .rename_axis("outcome")
        .reset_index(name="count")
    )
    outcome_prev["prevalence_pct"] = (100.0 * outcome_prev["count"] / len(df)).round(2)

    transactions = transactions.fillna(False).astype(bool)
    transactions = transactions.loc[:, transactions.any(axis=0)]

    summary = {
        "outcome_col": outcome_col,
        "date_col": date_col,
        "weather_numeric_cols": weather_numeric_cols,
        "weather_discretized_count": weather_discretized,
        "moon_cols": moon_cols,
        "moon_numeric_count": moon_numeric_count,
        "moon_categorical_count": moon_categorical_count,
    }

    exposure_items = set().union(weather_items, moon_items, temporal_items)

    return {
        "transactions": transactions,
        "exposure_items": exposure_items,
        "outcome_items": outcome_items,
        "summary": summary,
        "outcome_prevalence": outcome_prev,
    }


def _postprocess_rules(rules_df: pd.DataFrame) -> pd.DataFrame:
    if rules_df.empty:
        return rules_df
    out = rules_df.copy()
    out["antecedent_len"] = out["antecedents"].apply(len)
    out["consequent_len"] = out["consequents"].apply(len)
    out["antecedent_str"] = out["antecedents"].apply(itemset_to_str)
    out["consequent_str"] = out["consequents"].apply(itemset_to_str)
    out["rule_label"] = out["antecedent_str"] + " -> " + out["consequent_str"]
    out["composite_score"] = out["lift"] * out["confidence"]
    out["rule_key"] = out.apply(
        lambda r: rule_key(r["antecedents"], r["consequents"]), axis=1
    )
    return out


def run_apriori(
    transactions: pd.DataFrame,
    exposure_items: Set[str],
    outcome_items: Set[str],
    min_support: float = DEFAULT_MIN_SUPPORT,
    min_confidence: float = MIN_CONFIDENCE,
    min_lift: float = MIN_LIFT,
) -> Dict[str, pd.DataFrame]:
    if transactions.empty:
        return {
            "itemsets": pd.DataFrame(),
            "rules_all": pd.DataFrame(),
            "rules_filtered": pd.DataFrame(),
        }

    itemsets = apriori(
        transactions,
        min_support=min_support,
        use_colnames=True,
        max_len=3,
        low_memory=True,
    )

    if itemsets.empty:
        return {
            "itemsets": itemsets,
            "rules_all": pd.DataFrame(),
            "rules_filtered": pd.DataFrame(),
        }

    rules_all = association_rules(itemsets, metric="confidence", min_threshold=0.0)
    rules_all = _postprocess_rules(rules_all)
    if rules_all.empty:
        return {
            "itemsets": itemsets,
            "rules_all": rules_all,
            "rules_filtered": pd.DataFrame(),
        }

    rules_all = rules_all.sort_values(
        ["lift", "confidence", "support"], ascending=[False, False, False]
    ).reset_index(drop=True)

    filtered = rules_all[
        rules_all["antecedents"].apply(lambda s: len(s) > 0 and set(s).issubset(exposure_items))
        & rules_all["consequents"].apply(lambda s: len(s) > 0 and set(s).issubset(outcome_items))
    ].copy()

    filtered = filtered[
        (filtered["lift"] >= min_lift)
        & (filtered["confidence"] >= min_confidence)
        & (filtered["support"] >= min_support)
    ].copy()

    filtered = filtered.sort_values(
        ["lift", "confidence", "support"], ascending=[False, False, False]
    ).reset_index(drop=True)

    return {
        "itemsets": itemsets,
        "rules_all": rules_all,
        "rules_filtered": filtered,
    }


def prune_rules(
    rules_df: pd.DataFrame,
    conf_diff_max: float = PRUNE_CONF_DIFF,
    lift_diff_max: float = PRUNE_LIFT_DIFF,
) -> pd.DataFrame:
    if rules_df.empty:
        return rules_df.copy()

    work = rules_df.copy().reset_index(drop=True)
    work["consequent_key"] = work["consequents"].apply(lambda s: tuple(sorted(s)))
    work["antecedent_len"] = work["antecedents"].apply(len)

    keep_indices: List[int] = []
    for _, group in work.groupby("consequent_key", sort=False):
        group_sorted = group.sort_values(
            ["antecedent_len", "lift", "confidence"],
            ascending=[True, False, False],
        )
        kept_local: List[int] = []

        for idx, row in group_sorted.iterrows():
            ant_row = set(row["antecedents"])
            is_redundant = False
            for kept_idx in kept_local:
                ant_kept = set(work.at[kept_idx, "antecedents"])
                if ant_row > ant_kept:
                    conf_diff = abs(float(row["confidence"]) - float(work.at[kept_idx, "confidence"]))
                    lift_diff = abs(float(row["lift"]) - float(work.at[kept_idx, "lift"]))
                    if conf_diff < conf_diff_max and lift_diff < lift_diff_max:
                        is_redundant = True
                        break
            if not is_redundant:
                kept_local.append(idx)
        keep_indices.extend(kept_local)

    pruned = work.loc[sorted(set(keep_indices))].copy()
    pruned = pruned.sort_values(
        ["lift", "confidence", "support"], ascending=[False, False, False]
    ).reset_index(drop=True)
    pruned = pruned.drop(columns=["consequent_key"], errors="ignore")
    return pruned


def run_sensitivity(
    transactions: pd.DataFrame,
    exposure_items: Set[str],
    outcome_items: Set[str],
    supports: Sequence[float] = SENSITIVITY_SUPPORTS,
) -> pd.DataFrame:
    per_support: Dict[float, pd.DataFrame] = {}

    for sup in supports:
        res = run_apriori(
            transactions=transactions,
            exposure_items=exposure_items,
            outcome_items=outcome_items,
            min_support=float(sup),
            min_confidence=MIN_CONFIDENCE,
            min_lift=MIN_LIFT,
        )
        pruned = prune_rules(res["rules_filtered"])
        per_support[float(sup)] = pruned

    all_rule_keys: Set[str] = set()
    for _, df_sup in per_support.items():
        if not df_sup.empty:
            all_rule_keys.update(df_sup["rule_key"].tolist())

    rows: List[Dict[str, object]] = []
    for key in sorted(all_rule_keys):
        row: Dict[str, object] = {"rule_key": key}
        present_count = 0
        for sup in supports:
            tag = f"{float(sup):.2f}".replace(".", "_")
            df_sup = per_support[float(sup)]
            hit = df_sup[df_sup["rule_key"] == key]
            present = not hit.empty
            row[f"present_sup_{tag}"] = present
            if present:
                present_count += 1
                row[f"lift_sup_{tag}"] = float(hit.iloc[0]["lift"])
                row[f"confidence_sup_{tag}"] = float(hit.iloc[0]["confidence"])
                row[f"support_sup_{tag}"] = float(hit.iloc[0]["support"])
            else:
                row[f"lift_sup_{tag}"] = np.nan
                row[f"confidence_sup_{tag}"] = np.nan
                row[f"support_sup_{tag}"] = np.nan
            if "antecedent_str" not in row and present:
                row["antecedent_str"] = hit.iloc[0]["antecedent_str"]
                row["consequent_str"] = hit.iloc[0]["consequent_str"]
        row["n_thresholds_present"] = present_count
        row["is_stable_all_thresholds"] = present_count == len(supports)
        rows.append(row)

    stability = pd.DataFrame(rows)
    if not stability.empty:
        stability = stability.sort_values(
            ["n_thresholds_present", "is_stable_all_thresholds"],
            ascending=[False, False],
        ).reset_index(drop=True)
    return stability


def plot_top15_barplot(rules_df: pd.DataFrame, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(12, 8))
    if rules_df.empty:
        ax.text(0.5, 0.5, "No rules available", ha="center", va="center")
        ax.set_title("Top 15 Rules by Composite Score")
        ax.set_xlabel("Composite score (lift * confidence)")
        ax.set_ylabel("Rule")
        fig.tight_layout()
        fig.savefig(output_path, dpi=200)
        plt.close(fig)
        return

    top = rules_df.head(15).copy()
    top = top.iloc[::-1]
    ax.barh(top["rule_label"], top["composite_score"], color="#2E6E8E")
    ax.set_title("Top 15 Rules by Composite Score")
    ax.set_xlabel("Composite score (lift * confidence)")
    ax.set_ylabel("Rule")
    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)


def plot_lift_vs_confidence(rules_df: pd.DataFrame, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 7))
    if rules_df.empty:
        ax.text(0.5, 0.5, "No rules available", ha="center", va="center")
        ax.set_title("Lift vs Confidence")
        ax.set_xlabel("Confidence")
        ax.set_ylabel("Lift")
        fig.tight_layout()
        fig.savefig(output_path, dpi=200)
        plt.close(fig)
        return

    sizes = (rules_df["support"].clip(lower=0.001) * 2500).to_numpy()
    ax.scatter(
        rules_df["confidence"],
        rules_df["lift"],
        s=sizes,
        alpha=0.65,
        c="#3A9D23",
        edgecolors="black",
        linewidths=0.4,
    )
    ax.set_title("Lift vs Confidence (size ~ support)")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Lift")

    label_df = rules_df.head(10)
    for _, r in label_df.iterrows():
        ax.text(
            float(r["confidence"]) + 0.003,
            float(r["lift"]) + 0.003,
            r["rule_label"][:60],
            fontsize=8,
        )

    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)


def plot_rules_by_outcome(rules_df: pd.DataFrame, output_path: Path) -> None:
    if rules_df.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.text(0.5, 0.5, "No rules available", ha="center", va="center")
        ax.set_title("Top Rules by Outcome")
        fig.tight_layout()
        fig.savefig(output_path, dpi=200)
        plt.close(fig)
        return

    outcomes = sorted(rules_df["consequent_str"].unique().tolist())
    n = len(outcomes)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4.5 * nrows))
    axes = np.array(axes).reshape(-1)

    for i, outcome in enumerate(outcomes):
        ax = axes[i]
        subset = rules_df[rules_df["consequent_str"] == outcome].head(5).copy()
        subset = subset.iloc[::-1]
        ax.barh(subset["antecedent_str"], subset["lift"], color="#A24D2F")
        ax.set_title(f"{outcome}")
        ax.set_xlabel("Lift")
        ax.set_ylabel("Antecedent")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    fig.suptitle("Top 5 Rules by Lift for Each Outcome", y=0.995)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)


def plot_rules_network(rules_df: pd.DataFrame, output_path: Path, top_n: int = 30) -> None:
    fig, ax = plt.subplots(figsize=(12, 8))
    if rules_df.empty:
        ax.text(0.5, 0.5, "No rules available", ha="center", va="center")
        ax.set_title("Weather Items to Outcome Items Network")
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(output_path, dpi=200)
        plt.close(fig)
        return

    top = rules_df.head(top_n).copy()
    weather_nodes: Set[str] = set()
    outcome_nodes: Set[str] = set()
    edges: List[Tuple[str, str, float]] = []

    for _, r in top.iterrows():
        ants = list(r["antecedents"])
        cons = list(r["consequents"])
        for a in ants:
            weather_nodes.add(a)
            for c in cons:
                outcome_nodes.add(c)
                edges.append((a, c, float(r["lift"])))

    weather_nodes = set(sorted(weather_nodes))
    outcome_nodes = set(sorted(outcome_nodes))
    wy = {n: i for i, n in enumerate(sorted(weather_nodes))}
    oy = {n: i for i, n in enumerate(sorted(outcome_nodes))}

    max_w = max(1, len(wy) - 1)
    max_o = max(1, len(oy) - 1)

    weather_pos = {n: (0.15, 1.0 - i / max_w) for n, i in wy.items()}
    outcome_pos = {n: (0.85, 1.0 - i / max_o) for n, i in oy.items()}

    for a, c, lift in edges:
        x1, y1 = weather_pos[a]
        x2, y2 = outcome_pos[c]
        ax.plot([x1, x2], [y1, y2], color="#808080", alpha=min(0.9, 0.2 + (lift / 5.0)), linewidth=1.0)

    for n, (x, y) in weather_pos.items():
        ax.scatter([x], [y], s=40, color="#3366AA")
        ax.text(x - 0.02, y, n[:30], fontsize=7, ha="right", va="center")

    for n, (x, y) in outcome_pos.items():
        ax.scatter([x], [y], s=60, color="#AA3333")
        ax.text(x + 0.02, y, n[:30], fontsize=7, ha="left", va="center")

    ax.set_title("Weather/Time Items to Outcome Items Network (Top Rules)")
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.05, 1.05)
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)


def export_itemsets(itemsets: pd.DataFrame) -> pd.DataFrame:
    if itemsets.empty:
        return itemsets.copy()
    out = itemsets.copy()
    out["itemset"] = out["itemsets"].apply(itemset_to_str)
    out["item_count"] = out["itemsets"].apply(len)
    out = out.drop(columns=["itemsets"])
    cols = ["itemset", "item_count", "support"] + [c for c in out.columns if c not in {"itemset", "item_count", "support"}]
    out = out[cols]
    return out


def export_rules(rules: pd.DataFrame) -> pd.DataFrame:
    if rules.empty:
        return rules.copy()
    out = rules.copy()
    out["antecedents"] = out["antecedents"].apply(itemset_to_str)
    out["consequents"] = out["consequents"].apply(itemset_to_str)
    preferred_cols = [
        "rule_key",
        "rule_label",
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift",
        "leverage",
        "conviction",
        "composite_score",
        "antecedent support",
        "consequent support",
        "antecedent_len",
        "consequent_len",
    ]
    cols = [c for c in preferred_cols if c in out.columns] + [c for c in out.columns if c not in preferred_cols]
    return out[cols]


def build_report_markdown(
    n_rows: int,
    n_items: int,
    outcome_col: str,
    outcome_prev: pd.DataFrame,
    discretization_summary: Dict[str, object],
    top_rules: pd.DataFrame,
    output_paths: Dict[str, Path],
) -> str:
    lines: List[str] = []
    lines.append("# Apriori Weather-Outcome Pattern Mining Report")
    lines.append("")
    lines.append("## Dataset Overview")
    lines.append(f"- Number of rows (presentations): {n_rows}")
    lines.append(f"- Number of boolean transaction items: {n_items}")
    lines.append(f"- Outcome column used: `{outcome_col}`")
    lines.append("")

    lines.append("## Outcome Prevalence")
    lines.append(to_markdown_table(outcome_prev, ["outcome", "count", "prevalence_pct"]))
    lines.append("")

    lines.append("## Discretization Strategy")
    lines.append("- Continuous weather features discretized with tertiles (`pd.qcut`, `duplicates='drop'`) into low/mid/high.")
    lines.append("- Moon numeric features discretized with quartiles (`pd.qcut`, `duplicates='drop'`).")
    lines.append("- Moon categorical features one-hot encoded.")
    lines.append("- Temporal items encoded from date-derived month, weekday, and season when available.")
    lines.append("- Outcome categories encoded as boolean `outcome_*` items.")
    lines.append("")
    lines.append("### Discretization Counts")
    lines.append(f"- Weather numeric candidates: {len(discretization_summary.get('weather_numeric_cols', []))}")
    lines.append(f"- Weather features discretized: {discretization_summary.get('weather_discretized_count', 0)}")
    lines.append(f"- Moon columns detected: {len(discretization_summary.get('moon_cols', []))}")
    lines.append(f"- Moon numeric discretized: {discretization_summary.get('moon_numeric_count', 0)}")
    lines.append(f"- Moon categorical one-hot: {discretization_summary.get('moon_categorical_count', 0)}")
    lines.append("")

    lines.append("## Top Rules per Outcome Group")
    if top_rules.empty:
        lines.append("_No rules met thresholds after filtering and pruning._")
    else:
        grouped = top_rules.groupby("consequent_str", sort=False)
        for outcome, subset in grouped:
            lines.append(f"### {outcome}")
            take = subset.head(5).copy()
            table_df = take[["antecedent_str", "support", "confidence", "lift", "composite_score"]].copy()
            for col in ["support", "confidence", "lift", "composite_score"]:
                table_df[col] = table_df[col].map(lambda x: f"{x:.4f}")
            lines.append(
                to_markdown_table(
                    table_df.rename(columns={"antecedent_str": "antecedent"}),
                    ["antecedent", "support", "confidence", "lift", "composite_score"],
                )
            )
            lines.append("")

    lines.append("## Interpretation")
    if not top_rules.empty:
        exposure_bundles = top_rules["antecedent_str"].head(10).tolist()
        lines.append("- Exposure bundles identified (top examples):")
        for b in exposure_bundles:
            lines.append(f"  - {b}")
    else:
        lines.append("- Exposure bundles identified: none above thresholds in this run.")
    lines.append("- Lift interpretation: lift > 1 indicates co-occurrence above chance for the outcome rule.")
    lines.append("- Warning: exploratory association mining only, not causal inference.")
    lines.append("- Discretization limitation: quantile binning may hide within-bin gradients and threshold artifacts.")
    lines.append("- Collinearity caution: weather windows (day/mean/peak) are correlated and can inflate redundant patterns.")
    lines.append("")

    lines.append("## Suggested Next Step")
    lines.append("- Test top antecedent bundles in logistic regression as interaction/composite terms with proper validation.")
    lines.append("")

    lines.append("## Output Files")
    for logical_name, path in output_paths.items():
        lines.append(f"- {logical_name}: `{path}`")
    lines.append("")

    return "\n".join(lines)


def save_outputs(
    output_dir: Path,
    frequent_itemsets: pd.DataFrame,
    rules_all: pd.DataFrame,
    rules_filtered_pruned: pd.DataFrame,
    rules_top50: pd.DataFrame,
    stability_summary: pd.DataFrame,
    report_markdown: str,
) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths: Dict[str, Path] = {}

    file_plan = {
        "frequent_itemsets.csv": export_itemsets(frequent_itemsets),
        "rules_all.csv": export_rules(rules_all),
        "rules_weather_to_outcome_filtered.csv": export_rules(rules_filtered_pruned),
        "rules_top50.csv": export_rules(rules_top50),
        "stability_summary.csv": stability_summary,
    }

    for filename, df in file_plan.items():
        path = resolve_output_path(output_dir, filename)
        df.to_csv(path, index=False)
        paths[filename] = path

    top15_plot_path = resolve_output_path(output_dir, "rules_top15_barplot.png")
    plot_top15_barplot(rules_top50, top15_plot_path)
    paths["rules_top15_barplot.png"] = top15_plot_path

    scatter_plot_path = resolve_output_path(output_dir, "lift_vs_confidence.png")
    plot_lift_vs_confidence(rules_filtered_pruned, scatter_plot_path)
    paths["lift_vs_confidence.png"] = scatter_plot_path

    outcome_plot_path = resolve_output_path(output_dir, "rules_by_outcome.png")
    plot_rules_by_outcome(rules_filtered_pruned, outcome_plot_path)
    paths["rules_by_outcome.png"] = outcome_plot_path

    network_plot_path = resolve_output_path(output_dir, "rules_network.png")
    plot_rules_network(rules_filtered_pruned, network_plot_path)
    paths["rules_network.png"] = network_plot_path

    report_path = resolve_output_path(output_dir, "apriori_report.md")
    report_path.write_text(report_markdown, encoding="utf-8")
    paths["apriori_report.md"] = report_path

    return paths


def main() -> None:
    logger = setup_logging()
    np.random.seed(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = load_data(ENRICHED_PATH)
    logger.info("Loaded enriched dataset shape: %s", df.shape)

    built = build_transactions(df)
    transactions: pd.DataFrame = built["transactions"]
    exposure_items: Set[str] = built["exposure_items"]
    outcome_items: Set[str] = built["outcome_items"]
    summary: Dict[str, object] = built["summary"]
    outcome_prev: pd.DataFrame = built["outcome_prevalence"]

    logger.info("Transaction matrix shape: %s", transactions.shape)
    logger.info("Outcome distribution (%s):", summary["outcome_col"])
    for _, row in outcome_prev.iterrows():
        logger.info("  %s | count=%s | prevalence=%.2f%%", row["outcome"], int(row["count"]), float(row["prevalence_pct"]))

    apriori_default = run_apriori(
        transactions=transactions,
        exposure_items=exposure_items,
        outcome_items=outcome_items,
        min_support=DEFAULT_MIN_SUPPORT,
        min_confidence=MIN_CONFIDENCE,
        min_lift=MIN_LIFT,
    )

    frequent_itemsets = apriori_default["itemsets"]
    rules_all = apriori_default["rules_all"]
    rules_filtered = apriori_default["rules_filtered"]
    rules_filtered_pruned = prune_rules(rules_filtered)
    rules_top50 = rules_filtered_pruned.head(50).copy()

    stability_summary = run_sensitivity(
        transactions=transactions,
        exposure_items=exposure_items,
        outcome_items=outcome_items,
        supports=SENSITIVITY_SUPPORTS,
    )

    placeholder_paths = {
        "frequent_itemsets.csv": OUTPUT_DIR / "frequent_itemsets.csv",
        "rules_all.csv": OUTPUT_DIR / "rules_all.csv",
        "rules_weather_to_outcome_filtered.csv": OUTPUT_DIR / "rules_weather_to_outcome_filtered.csv",
        "rules_top50.csv": OUTPUT_DIR / "rules_top50.csv",
        "stability_summary.csv": OUTPUT_DIR / "stability_summary.csv",
        "rules_top15_barplot.png": OUTPUT_DIR / "rules_top15_barplot.png",
        "lift_vs_confidence.png": OUTPUT_DIR / "lift_vs_confidence.png",
        "rules_by_outcome.png": OUTPUT_DIR / "rules_by_outcome.png",
        "rules_network.png": OUTPUT_DIR / "rules_network.png",
        "apriori_report.md": OUTPUT_DIR / "apriori_report.md",
    }

    report_markdown = build_report_markdown(
        n_rows=len(df),
        n_items=transactions.shape[1],
        outcome_col=str(summary["outcome_col"]),
        outcome_prev=outcome_prev,
        discretization_summary=summary,
        top_rules=rules_filtered_pruned,
        output_paths=placeholder_paths,
    )

    saved_paths = save_outputs(
        output_dir=OUTPUT_DIR,
        frequent_itemsets=frequent_itemsets,
        rules_all=rules_all,
        rules_filtered_pruned=rules_filtered_pruned,
        rules_top50=rules_top50,
        stability_summary=stability_summary,
        report_markdown=report_markdown,
    )


    report_markdown = build_report_markdown(
        n_rows=len(df),
        n_items=transactions.shape[1],
        outcome_col=str(summary["outcome_col"]),
        outcome_prev=outcome_prev,
        discretization_summary=summary,
        top_rules=rules_filtered_pruned,
        output_paths=saved_paths,
    )
    saved_paths["apriori_report.md"].write_text(report_markdown, encoding="utf-8")

    logger.info("Apriori pipeline finished. Outputs saved under: %s", OUTPUT_DIR)
    for name, path in saved_paths.items():
        logger.info("  %s -> %s", name, path)


if __name__ == "__main__":
    main()
